#  Multi-Agent Testing
In this notebook, we will:
1. Initialize the LLM (Groq) and load our prepared databases.
2. Test the Orchestrator Agent to ensure it correctly routes queries.
3. Test the NLP-to-SQL Agent to query the patient database.
4. Test the RAG Agent to retrieve hospital policies.

## Setup & Load Components

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


d:\Healthcare_Query_Assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\AMAN\AppData\Local\Temp\ipykernel_31268\2374475744.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [2]:
# Load environment variables from the parent directory
load_dotenv('../.env')

True

In [3]:
# Initialize LLM (Using Groq for speed and to avoid SQL blocking)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env file!")

In [4]:
llm = ChatOpenAI(
    model="llama-3.3-70b-versatile",
    base_url="https://api.groq.com/openai/v1",
    api_key=GROQ_API_KEY,
    temperature=0
)
print("✅ LLM Initialized (Groq)")

✅ LLM Initialized (Groq)


In [5]:
# Load SQL Database
db = SQLDatabase.from_uri("sqlite:///../healthcare.db")
print("✅ SQL Database Loaded")

✅ SQL Database Loaded


In [6]:
# Load Vector Store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.load_local("../faiss_index", embeddings, allow_dangerous_deserialization=True)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("✅ FAISS Vector Store Loaded")

C:\Users\AMAN\AppData\Local\Temp\ipykernel_31268\3368153558.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3184.01it/s]


✅ FAISS Vector Store Loaded


## Test the Orchestrator / Router

In [7]:
# Define the Router Prompt
router_prompt = ChatPromptTemplate.from_template(
    """You are an expert routing agent. Analyze the user's query and determine which system should handle it.
    
    - Output 'SQL' if the query is about patient records, medical conditions, billing amounts, doctors, admissions, or specific patient data.
    - Output 'RAG' if the query is about hospital rules, policies, procedures, insurance approvals, or general guidelines.
    - Output 'UNKNOWN' if the query is unrelated to healthcare.
    
    User Query: {query}
    Output only 'SQL', 'RAG', or 'UNKNOWN':"""
)
router_chain = router_prompt | llm | StrOutputParser()

# Test the Router
test_queries = [
    "How many diabetic patients were admitted last month?",
    "What is the hospital discharge policy?",
    "What is the weather like today?"
]

print("--- Testing Orchestrator (Router) ---")
for q in test_queries:
    route = router_chain.invoke({"query": q}).strip().upper()
    print(f"Query: '{q}' \nRouted to: {route}\n")

--- Testing Orchestrator (Router) ---
Query: 'How many diabetic patients were admitted last month?' 
Routed to: SQL

Query: 'What is the hospital discharge policy?' 
Routed to: RAG

Query: 'What is the weather like today?' 
Routed to: UNKNOWN



## Test the NLP-to-SQL Agent

In [8]:
# Create the SQL Agent with a custom prefix to handle messy data casing
sql_agent = create_sql_agent(
    llm=llm,
    db=db,
    agent_type="openai-tools",
    verbose=False,
    top_k=10,
    prefix="""You are an agent designed to interact with a SQL database.
    Given an input question, create a syntactically correct SQL query to run.
    IMPORTANT: The database contains messy, inconsistently capitalized names (e.g., 'daVId bEcKeR'). 
    When querying names or text fields, ALWAYS use LOWER(column_name) = LOWER('search_term') to ensure you find the records.
    Unless the user specifies a specific number of examples to obtain, query for at most 5 results.
    Never query for all columns from a table. You must query only the columns that are needed to answer the question.
    Wrap each column name in double quotes (") to denote them as delimited identifiers.
    Format your final response as a clear, conversational answer."""
)

# 🛡️ Safety Patch: Prevent empty string errors from crashing the LLM
def safe_output(method):
    def wrapper(*args, **kwargs):
        try:
            result = method(*args, **kwargs)
        except Exception as e:
            return f"Error executing tool: {str(e)}"
        if result is None or result == "" or result == [] or result == ():
            return "No results found."
        if isinstance(result, str) and not result.strip():
            return "No results found."
        return result
    return wrapper

for tool in sql_agent.tools:
    if hasattr(tool, '_run'):
        tool._run = safe_output(tool._run)

# Test the SQL Agent
sql_query = "Show me the average billing amount for patients with O+ blood group."
print(f"--- Testing SQL Agent ---")
print(f"Query: {sql_query}\n")

sql_response = sql_agent.invoke({"input": sql_query})["output"]
print(f"Agent Response:\n{sql_response}")

--- Testing SQL Agent ---
Query: Show me the average billing amount for patients with O+ blood group.

Agent Response:
The average billing amount for patients with O+ blood group is not available as the query returned no results.


## Test the RAG Agent

In [10]:
# Define the RAG Prompt
rag_prompt = ChatPromptTemplate.from_template(
    """You are a helpful hospital policy assistant. Answer the user's question based ONLY on the following context. 
    If the answer is not in the context, say "I don't have that information in the policy documents."
    
    Context: {context}
    Question: {question}
    """
)
rag_chain = rag_prompt | llm | StrOutputParser()

# Test the RAG Agent
rag_query = "Is prior insurance approval required for an MRI scan?"
print(f"--- Testing RAG Agent ---")
print(f"Query: {rag_query}\n")

# Retrieve context
docs = retriever.invoke(rag_query)
context = "\n\n".join([doc.page_content for doc in docs])

# Generate response
rag_response = rag_chain.invoke({"context": context, "question": rag_query})
print(f"Retrieved Context Snippet:\n{context[:150]}...\n")
print(f"Agent Response:\n{rag_response}")

--- Testing RAG Agent ---
Query: Is prior insurance approval required for an MRI scan?

Retrieved Context Snippet:
Insurance Approval Policy: Prior approval from the insurance provider is strictly required for all surgical procedures and advanced imaging (MRI/CT sc...

Agent Response:
Yes, prior approval from the insurance provider is strictly required for an MRI scan, as it is considered an advanced imaging procedure.


In [11]:
import time

print("--- 📊 Phase 5: Latency & Accuracy Evaluation ---")

test_cases = [
    {"query": "How many patients have Diabetes?", "expected_route": "SQL"},
    {"query": "What is the maximum number of visitors in the ER?", "expected_route": "RAG"},
    {"query": "What is the capital of France?", "expected_route": "UNKNOWN"}
]

accurate_routes = 0
total_latency = 0

for case in test_cases:
    start_time = time.time()
    
    # 1. Route
    route = router_chain.invoke({"query": case['query']}).strip().upper()
    
    # 2. Execute (Simplified for testing)
    if "SQL" in route:
        response = sql_agent.invoke({"input": case['query']})["output"]
    elif "RAG" in route:
        docs = retriever.invoke(case['query'])
        context = "\n\n".join([doc.page_content for doc in docs])
        response = rag_chain.invoke({"context": context, "question": case['query']})
    else:
        response = "Fallback response."
        
    end_time = time.time()
    latency = end_time - start_time
    total_latency += latency
    
    # Check accuracy
    if case['expected_route'] in route:
        accurate_routes += 1
        status = "✅ Accurate"
    else:
        status = "❌ Inaccurate"
        
    print(f"Query: '{case['query']}' | Route: {route} | Latency: {latency:.2f}s | {status}")

avg_latency = total_latency / len(test_cases)
accuracy_pct = (accurate_routes / len(test_cases)) * 100

print("-" * 50)
print(f"📈 Overall Routing Accuracy: {accuracy_pct}%")
print(f"⏱️ Average Response Latency: {avg_latency:.2f} seconds")

--- 📊 Phase 5: Latency & Accuracy Evaluation ---
Query: 'How many patients have Diabetes?' | Route: SQL | Latency: 3.72s | ✅ Accurate
Query: 'What is the maximum number of visitors in the ER?' | Route: RAG | Latency: 1.02s | ✅ Accurate
Query: 'What is the capital of France?' | Route: UNKNOWN | Latency: 0.20s | ✅ Accurate
--------------------------------------------------
📈 Overall Routing Accuracy: 100.0%
⏱️ Average Response Latency: 1.64 seconds
